# 🔴 Solution: RoPE

In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def apply_rotary_pos_emb(x: torch.Tensor, pos: torch.Tensor) -> torch.Tensor:
    # x: (batch, num_heads, seq_len, head_dim)
    # pos: (batch, seq_len) position indices
    
    batch, num_heads, seq_len, head_dim = x.shape
    
    # Compute frequencies: theta_i = 1 / (10000^(2i/d))
    freqs = 1.0 / (10000 ** (torch.arange(0, head_dim, 2, device=x.device).float() / head_dim))
    
    # pos: (batch, seq_len) -> (batch, seq_len, 1)
    pos = pos.unsqueeze(-1).float()
    
    # angles: (batch, seq_len, head_dim/2)
    angles = pos * freqs.unsqueeze(0).unsqueeze(0)
    
    # cos and sin
    cos = torch.cos(angles)  # (batch, seq_len, head_dim/2)
    sin = torch.sin(angles)
    
    # Reshape x for rotation: split into pairs
    x1 = x[..., 0::2]  # (batch, num_heads, seq_len, head_dim/2)
    x2 = x[..., 1::2]
    
    # Apply rotation
    cos = cos.unsqueeze(1)  # (batch, 1, seq_len, head_dim/2)
    sin = sin.unsqueeze(1)
    
    x_rotated_1 = x1 * cos - x2 * sin
    x_rotated_2 = x1 * sin + x2 * cos
    
    # Interleave back
    x_out = torch.stack([x_rotated_1, x_rotated_2], dim=-1).flatten(-2)
    
    return x_out

In [ ]:
# Verify
x = torch.randn(2, 4, 8, 16)
pos = torch.arange(8).unsqueeze(0).expand(2, -1)
out = apply_rotary_pos_emb(x, pos)
print(f"Output shape: {out.shape}")

In [ ]:
from torch_judge import check
check("rope")